In [10]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")

import jax
jax.config.update("jax_enable_x64", True)

from pyscf import gto, scf, cc
from pyscf.data import elements

import numpy as np

from afqmc.corr_sample import integral, launch_afqmc

####  test monomers ####
a = 1.20577 # bond length in a cluster
d = 100 # distance between each cluster
unit = 'A' # unit of length
na = 2 # size of a cluster (monomer)
nc = 1 # set as integer multiple of monomers
spin = 2 # spin per monomer
frozen = 0 # frozen orbital per monomer
elmt = 'O'
basis = 'sto6g'
atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"
###########################

mol1 = gto.M(atom=atoms,
            basis=basis,
            verbose=4,
            unit=unit,
            symmetry=0,
            charge=0,
            spin=spin*nc,
            max_memory=40000,
            )

mf1 = scf.UHF(mol1).density_fit()
mf1.kernel()

stable = False
while not stable:
    print(f'mean-field stability test')
    if not stable:
        mo_i, _, stable,_ = mf1.stability(return_status=True)
        dm = mf1.make_rdm1(mo_i,mf1.mo_occ)
        mf1.kernel(dm0=dm)
    elif stable:
        print(f'HF Energy: {mf1.e_tot}, stability {stable}')
        break

mol2 = gto.M(atom=atoms,
            basis=basis,
            verbose=4,
            unit=unit,
            symmetry=0,
            charge=0,
            spin=spin*nc,
            max_memory=40000,
            )

mf2 = scf.UHF(mol2).density_fit()
mf2.kernel(dm0 = mf1.make_rdm1())

stable = False
while not stable:
    print(f'mean-field stability test')
    if not stable:
        mo_i, _, stable,_ = mf2.stability(return_status=True)
        dm = mf2.make_rdm1(mo_i,mf2.mo_occ)
        mf2.kernel(dm0=dm)
    elif stable:
        print(f'HF Energy: {mf2.e_tot}, stability {stable}')
        break

frozen = elements.chemcore(mol1)
nocc_a, nocc_b = np.count_nonzero(mf1.mo_occ[0]), np.count_nonzero(mf1.mo_occ[1])
print(f"mf1 energy = {mf1.e_tot:.8f} | mf2 energy = {mf2.e_tot:.8f}")
print("before Procruste")
print("Alpha frz Norm : ", np.linalg.norm(mf1.mo_coeff[0][:,:frozen] - mf2.mo_coeff[0][:,:frozen]))
print(" Beta frz Norm : ", np.linalg.norm(mf1.mo_coeff[1][:,:frozen] - mf2.mo_coeff[1][:,:frozen]))
print("Alpha occ Norm : ", np.linalg.norm(mf1.mo_coeff[0][:,frozen:nocc_a] - mf2.mo_coeff[0][:,frozen:nocc_a]))
print(" Beta occ Norm : ", np.linalg.norm(mf1.mo_coeff[1][:,frozen:nocc_b] - mf2.mo_coeff[1][:,frozen:nocc_b]))
print("Alpha vir Norm : ", np.linalg.norm(mf1.mo_coeff[0][:,nocc_a:] - mf2.mo_coeff[0][:,nocc_a:]))
print(" Beta vir Norm : ", np.linalg.norm(mf1.mo_coeff[1][:,nocc_b:] - mf2.mo_coeff[1][:,nocc_b:]))
mf2.mo_coeff = integral.match_mo(mf1, mf2, frozen=frozen)
print("Alpha frz Norm : ", np.linalg.norm(mf1.mo_coeff[0][:,:frozen] - mf2.mo_coeff[0][:,:frozen]))
print(" Beta frz Norm : ", np.linalg.norm(mf1.mo_coeff[1][:,:frozen] - mf2.mo_coeff[1][:,:frozen]))
print("Alpha occ Norm : ", np.linalg.norm(mf1.mo_coeff[0][:,frozen:nocc_a] - mf2.mo_coeff[0][:,frozen:nocc_a]))
print(" Beta occ Norm : ", np.linalg.norm(mf1.mo_coeff[1][:,frozen:nocc_b] - mf2.mo_coeff[1][:,frozen:nocc_b]))
print("Alpha vir Norm : ", np.linalg.norm(mf1.mo_coeff[0][:,nocc_a:] - mf2.mo_coeff[0][:,nocc_a:]))
print(" Beta vir Norm : ", np.linalg.norm(mf1.mo_coeff[1][:,nocc_b:] - mf2.mo_coeff[1][:,nocc_b:]))

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-35-generic', version='#35~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue May 26 19:30:42 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Sat Jul  4 17:03:30 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 16
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 2
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = A
[INPUT] Symbol           X                Y                Z      uni

In [3]:
S = mf1.get_ovlp()
nb = np.count_nonzero(mf1.mo_occ[1])   # beta occupied count

# 1) Is there a near-degeneracy at the beta Fermi level?
eb1, eb2 = mf1.mo_energy[1], mf2.mo_energy[1]
print("beta gap, run1:", eb1[nb] - eb1[nb-1])
print("beta gap, run2:", eb2[nb] - eb2[nb-1])
print("beta HOMO/LUMO run1:", eb1[nb-2:nb+2])
print("beta HOMO/LUMO run2:", eb2[nb-2:nb+2])

# 2) Do the two beta OCCUPIED subspaces actually coincide?
#    singular values = cosines of principal angles; all ~1 => same subspace
o1, o2 = mf1.mo_coeff[1][:, :nb], mf2.mo_coeff[1][:, :nb]
sv = np.linalg.svd(o1.conj().T @ S @ o2, compute_uv=False)
print("beta occ-subspace overlap singular values:", np.round(sv, 5))

# 3) Did the two runs land on the same beta density at all?
db1 = mf1.make_rdm1()[1]
db2 = mf2.make_rdm1()[1]
print("beta density difference norm:", np.linalg.norm(db1 - db2))

beta gap, run1: 0.7472728740809165
beta gap, run2: 0.747251957828847
beta HOMO/LUMO run1: [-0.46845136 -0.46845136  0.27882152  0.27882152]
beta HOMO/LUMO run2: [-0.46844123 -0.46844123  0.27881072  0.27881073]
beta occ-subspace overlap singular values: [1.      1.      1.      1.      1.      0.99354 0.99354]
beta density difference norm: 0.2296255976459437
